In [23]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
import tensorflow as tf

In [24]:
# Cargar el modelo preentrenado YOLOv8
modelo = YOLO("yolov8n.pt")
modeloViolencia = tf.keras.models.load_model('../AlertGuard/modelo_guardado.keras')
classes = ['no_violencia', 'violencia']
print("Modelo cargado correctamente")

Modelo cargado correctamente


In [25]:
# Ruta de la carpeta con videos
carpeta_videos = "../videos" # Cambia esta ruta a tu carpeta con videos
frames_personas = "../frames_personas" # Carpeta para guardar los frames con personas
# Obtener la lista de videos en la carpeta
videos = [os.path.abspath(os.path.join(carpeta_videos, f)) for f in os.listdir(carpeta_videos) 
          if f.endswith(('.mp4', '.avi', '.mov'))]
print("Videos encontrados:", videos)

# Crear la carpeta para guardar los frames con personas
if not os.path.exists(frames_personas):
    os.makedirs(frames_personas)

Videos encontrados: ['c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\aRobbery020_x264.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\Burglary001_x264.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\Burglary002_x264.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\Burglary003_x264.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\Burglary004_x264.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\Burglary006_x264.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\Burglary007_x264.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\Burglary008_x264.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\Burglary009_x264.mp4', 'c:\\Us

In [ ]:
def mostrar_video_con_yolo(video_path):
    cap = cv2.VideoCapture(video_path)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break  # Si no hay más frames, salir del bucle

        # Aplicar YOLO para detección de personas
        results = modelo(frame)
        for r in results:
            for box in r.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cls = int(box.cls[0])

                # Filtrar solo personas (Clase 0 en COCO)
                if cls == 0:
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    # Guardar el recuadro de la persona en un archivo de imagen
                    frame_persona = frame[y1:y2, x1:x2]
                    # Darle a la imágen un tamaño fijo
                    frame_persona = cv2.resize(frame_persona, (128, 128))
                    img = frame_persona
                    cv2.imwrite(os.path.join(frames_personas, f"{os.path.basename(video_path)}_{x1}_{y1}.jpg"), frame_persona)
                    #Determinar si hay violencia usando el modelo de violencia
                    # Cargar la imagen
                    img = img.astype(np.float32) / 255.0
                    img = np.expand_dims(img, axis=0)
                    # Predecir la clase de la imagen
                    pred = modeloViolencia(img)
                    # Obtener la clase con mayor probabilidad
                    pred_class = classes[np.argmax(pred)]

                    # Dibujar el texto de la clase y la probabilidad en la imagen
                    cv2.putText(frame, f"Violencia: {pred_class} ({np.max(pred):.2f})", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                    

        # Mostrar el frame con detecciones
        cv2.imshow("YOLO - Detección de Personas", frame)

        # Salir con la tecla 'q'
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

# Reproducir los videos en vivo con detección de personas
mostrar_video_con_yolo(videos[0])


0: 480x640 1 bottle, 1 refrigerator, 86.1ms
Speed: 1.8ms preprocess, 86.1ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 bottle, 1 refrigerator, 95.9ms
Speed: 4.0ms preprocess, 95.9ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 bottle, 1 refrigerator, 85.9ms
Speed: 2.8ms preprocess, 85.9ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 bottle, 1 refrigerator, 92.8ms
Speed: 3.9ms preprocess, 92.8ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 bottle, 1 refrigerator, 80.5ms
Speed: 3.4ms preprocess, 80.5ms inference, 9.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 bottle, 1 refrigerator, 91.3ms
Speed: 2.7ms preprocess, 91.3ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 bottle, 1 refrigerator, 87.5ms
Speed: 0.0ms preprocess, 87.5ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 48